# Módulo 6: Delta Sharing — Compartir Datos Sin Moverlos

## Objetivos de aprendizaje
- Entender por qué Delta Sharing cambia las reglas del juego
- Conocer la arquitectura: Share, Schema, Table, Recipient
- Configurar Delta Sharing como proveedor (outbound)
- Consumir un share como recipient (inbound)
- Implementar seguridad, auditoría y patrones multi-cloud
- Preparar el terreno para la integración SAP del Módulo 7

---
## 📅 Este módulo se trabaja en DOS clases

| Clase | Contenido | Secciones |
|---|---|---|
| **Jueves (C8)** — El lado del PROVIDER | Teoría, arquitectura, tipos de recipients, crear el share completo en el Trial | 6.1 — 6.4 + Lab Parte 1 |
| **Viernes (C9)** — El lado del RECIPIENT | Consumir el share, Time Travel y CDF cross-workspace, costos, auditoría, buenas prácticas | 6.5 — 6.10 + Lab Parte 2 |

---
## 6.1 ¿Qué es Delta Sharing y por qué cambia las reglas?

### El problema que resuelve

```
ANTES (sin Delta Sharing):
  SAP → exportar CSV → enviar por email/SFTP → importar → duplicar datos
  Resultado: datos desactualizados, sin governance, storage duplicado

CON Delta Sharing:
  SAP → Delta Lake → Share → Recipient accede en tiempo real
  Resultado: datos frescos, governance centralizado, zero-copy
```

### Características clave
- **Open protocol**: estándar REST, no requiere Databricks en el lado consumidor
- **Zero-copy**: los datos no se mueven ni replican
- **Cross-cloud**: compartir entre AWS, Azure y GCP sin restricciones
- **Governance**: Unity Catalog controla quién ve qué con auditoría completa
- **Bidireccional**: SAP puede enviar datos a Databricks Y recibirlos de vuelta

## Plan de la clase de hoy — Delta Sharing en la práctica

> **El recorrido completo en 1h 45min:**
> 1. **Conceptos** — qué es Delta Sharing y por qué cambia las reglas (10 min)
> 2. **Demo lado PROVEEDOR** — creamos share, recipient y compartimos las Gold SAP (30 min)
> 3. **Demo lado RECEPTOR** — consumimos `samples` como Delta Share real (20 min)
> 4. **Análisis Real vs Budget** — cruce de datos compartidos con Gold SAP (25 min)
> 5. **Conexión con BDC** — qué hace SAP automáticamente (20 min)

---

### ⚙️ Sobre el lab del lado receptor

Para conectar workspace-a-workspace via Delta Sharing externo se requiere que el **administrador del metastore** habilite `External OpenSharing` desde `accounts.cloud.databricks.com`. En el Free Edition con cuenta personal el metastore es `System user` y bloquea ese setup — exactamente el escenario donde **BDC Connect resuelve el problema**: SAP gestiona toda la configuración del metastore por ti.

Para hoy demostramos:
- **El lado proveedor completo** — con nuestro share `sap_analytics_share` y las Gold SAP reales
- **El lado receptor con `samples`** — Databricks comparte estos datasets via Delta Sharing real
- **El análisis cruzado** — Gold SAP × datos externos compartidos

> *"Esta es exactamente la experiencia BDC en producción — ustedes ven el lado proveedor con sus datos SAP y consumen data products externos. La única diferencia es que en BDC ese setup del metastore lo hace SAP automáticamente."*


---
## SECCIÓN 1 — Demo del PROVEEDOR (lo que ya hicimos)

Mostramos en pantalla el share que ya está configurado en el workspace.

### En el Catalog Explorer → OpenSharing → Shared by me

```
fpya_grupo_argos                          ← share creado en el trial
  └── budget.ventas_presupuesto           (presupuesto FP&A)
  └── referencia.tasa_cambio              (USD/EUR → COP)

sap_analytics_share                       ← share del Free Edition (este workspace)
  └── finance.financial_summary
  └── sales.monthly_kpis  
  └── customers.customer_360
```


In [0]:
# ── Ver los shares disponibles desde este workspace ──
print("=== SHARES configurados como PROVEEDOR ===\n")
spark.sql("SHOW SHARES").show(truncate=False)

In [0]:
# ── Ver el detalle del share principal ──
print("=== Detalle del share sap_analytics_share ===\n")
spark.sql("DESCRIBE SHARE sap_analytics_share").show(truncate=False)

In [0]:
# ── Ver las tablas dentro del share ──
print("=== Tablas compartidas con alias de negocio ===\n")
spark.sql("SHOW ALL IN SHARE sap_analytics_share").show(truncate=False)

### Lo que vieron en el Catalog Explorer

Cuando un cliente externo (SAP Analytics Cloud, otro Databricks, partner) recibe este share:
- Ve las tablas con los **alias de negocio**, no los nombres técnicos
- Lee los Parquet directamente — **zero-copy**
- Cada query usa **su propio cómputo**, no el de Summa
- Las nuevas filas en Gold aparecen automáticamente en su lado

Esto es lo que hace **SAP BDC** cuando publica un Data Product:
1. SAP crea el share en BDC Foundation Services
2. SAP genera un recipient para tu workspace Databricks
3. Tú lees las tablas como si fueran locales — pero los datos viven en el Object Store SAP


---
## SECCIÓN 2 — Demo del RECEPTOR (con samples reales)

El workspace Free Edition ya tiene **Delta Shares activos recibidos de Databricks**. Estos son shares de producción reales — exactamente el mecanismo que usa BDC.

### En el Catalog Explorer → Shares Received

```
samples                                   ← share recibido de Databricks
  ├── nyctaxi.trips                       (datos taxis NYC)
  ├── tpch.customer                       (TPC-H benchmark)
  ├── tpch.orders                         
  └── bakehouse.sales_customers           (caso retail)
```


In [0]:
# ── Ver el contenido del share samples ──
print("=== Tablas en samples.tpch (TPC-H benchmark) ===\n")
spark.sql("SHOW TABLES IN samples.tpch").show(truncate=False)

print("\n=== Tablas en samples.bakehouse ===\n")
spark.sql("SHOW TABLES IN samples.bakehouse").show(truncate=False)

In [0]:
# ── Consultar la tabla compartida — zero-copy en acción ──
print("=== Datos de samples.bakehouse.sales_customers (Delta Share real) ===\n")
spark.sql("""
    SELECT * FROM samples.bakehouse.sales_customers
    LIMIT 10
""").show(truncate=False)

In [0]:
# ── Métricas del share ──
print("=== ¿Cuántos registros tiene el share que estamos leyendo? ===\n")
spark.sql("""
    SELECT 
        COUNT(*) AS total_clientes,
        COUNT(DISTINCT city) AS ciudades_distintas,
        COUNT(DISTINCT state) AS estados_distintos
    FROM samples.bakehouse.sales_customers
""").show()

### 💡 Punto clave de la demo

Los datos de `samples.bakehouse.sales_customers` **no están en este workspace** — viven en el storage de Databricks. Cuando ejecutamos el `SELECT`:

1. Unity Catalog identifica que es una tabla via Delta Sharing
2. Solicita URLs firmadas al servidor Delta Sharing de Databricks  
3. Descarga solo los Parquet files necesarios (con partition pruning)
4. Ejecuta la query con NUESTRO cómputo

Esto es **idéntico** a lo que pasaría con un data product de SAP BDC.

| Esta demo | BDC en producción |
|-----------|------------------|
| samples shared por Databricks | Data Product shared por SAP |
| `samples.bakehouse.sales_customers` | `sap_bdc.sales.I_SalesOrder` |
| Nuestro cluster procesa | Nuestro cluster procesa |
| Datos viven en storage Databricks | Datos viven en Object Store SAP |
| Zero-copy en el límite del consumidor | Zero-copy en el límite del consumidor |


---
## SECCIÓN 3 — Análisis cruzado: Gold SAP × Datos Compartidos

Ahora hacemos el caso de negocio real: cruzar nuestras ventas SAP (datos locales) con datos compartidos via Delta Sharing.

### Caso 1: Análisis Real vs Budget

> **Contexto:** El área de FP&A publica el presupuesto via Delta Sharing.
> Nosotros (equipo de datos) lo cruzamos con las ventas reales SAP para varianza.

Para este caso usamos la tabla `budget_ventas` que **simularíamos** recibir de FP&A. Como la demostración externa requiere el setup del metastore admin, creamos la tabla localmente como si ya hubiera llegado por el share.


In [0]:
# ── Crear la tabla budget como si viniera del share FP&A ──
# En producción esta tabla aparecería automáticamente en
# fpya_shared.budget.ventas_presupuesto vía Delta Sharing

from pyspark.sql.types import *
from pyspark.sql import Row

budget_data = [
    Row(year=2024,month=12,sales_org="1000",currency="COP",budget_revenue=7890600.0, budget_orders=13,scenario="OPTIMISTA"),
    Row(year=2024,month=12,sales_org="1000",currency="USD",budget_revenue=3890500.5, budget_orders=9, scenario="BASE"),
    Row(year=2024,month=12,sales_org="1000",currency="EUR",budget_revenue=2987600.0, budget_orders=7, scenario="BASE"),
    Row(year=2024,month=12,sales_org="2000",currency="COP",budget_revenue=4567800.8, budget_orders=11,scenario="BASE"),
    Row(year=2024,month=12,sales_org="2000",currency="USD",budget_revenue=5345600.3, budget_orders=12,scenario="CONSERVADOR"),
    Row(year=2024,month=12,sales_org="2000",currency="EUR",budget_revenue=4678900.0, budget_orders=10,scenario="BASE"),
    Row(year=2024,month=11,sales_org="1000",currency="COP",budget_revenue=7234567.0, budget_orders=12,scenario="BASE"),
    Row(year=2024,month=11,sales_org="1000",currency="USD",budget_revenue=3567800.2, budget_orders=9, scenario="BASE"),
    Row(year=2024,month=11,sales_org="1000",currency="EUR",budget_revenue=2678900.5, budget_orders=7, scenario="OPTIMISTA"),
    Row(year=2024,month=11,sales_org="2000",currency="COP",budget_revenue=4234500.7, budget_orders=10,scenario="BASE"),
    Row(year=2024,month=11,sales_org="2000",currency="USD",budget_revenue=5023400.0, budget_orders=11,scenario="BASE"),
    Row(year=2024,month=11,sales_org="2000",currency="EUR",budget_revenue=4345600.3, budget_orders=10,scenario="BASE"),
]

df_budget = spark.createDataFrame(budget_data)

# Crear como tabla simulando que vino por Delta Sharing
spark.sql("CREATE SCHEMA IF NOT EXISTS laboratory_sap_dev.fpya_shared")
df_budget.write.mode("overwrite").saveAsTable("laboratory_sap_dev.fpya_shared.budget_ventas")

print(f"✅ Tabla budget_ventas simulada — {df_budget.count()} registros")
print("    (En producción esta tabla llegaría desde el share fpya_grupo_argos)")
df_budget.show(5)

In [0]:
# ── ANÁLISIS REAL VS BUDGET ──
# Cruzamos gold_sales_kpis (local SAP) con budget (desde "share" simulado)

print("=== Real vs Presupuesto · Diciembre 2024 ===\n")
spark.sql("""
    SELECT
        r.year,
        r.month,
        r.sales_org,
        r.currency,
        -- Ventas reales (gold local SAP)
        r.num_orders                                              AS ordenes_reales,
        ROUND(r.total_revenue, 2)                                 AS ventas_reales,
        -- Presupuesto (via Delta Sharing simulado)
        b.budget_orders                                           AS ordenes_budget,
        ROUND(b.budget_revenue, 2)                                AS ventas_budget,
        b.scenario                                                AS escenario,
        -- Análisis de varianza
        ROUND(r.total_revenue - b.budget_revenue, 2)              AS varianza,
        ROUND((r.total_revenue / b.budget_revenue - 1) * 100, 1)  AS varianza_pct,
        -- Semáforo
        CASE
            WHEN r.total_revenue >= b.budget_revenue              THEN 'CUMPLE'
            WHEN r.total_revenue >= b.budget_revenue * 0.90       THEN 'CERCA (>90%)'
            WHEN r.total_revenue >= b.budget_revenue * 0.75       THEN 'GAP MODERADO'
            ELSE                                                       'GAP CRITICO'
        END                                                       AS estado
    FROM laboratory_sap_dev.sap_course.gold_sales_kpis            r
    JOIN laboratory_sap_dev.fpya_shared.budget_ventas             b
      ON r.year      = b.year
     AND r.month     = b.month
     AND r.sales_org = b.sales_org
     AND r.currency  = b.currency
    WHERE r.year = 2024 AND r.month IN (11, 12)
    ORDER BY r.year DESC, r.month DESC, varianza_pct ASC
""").show(truncate=False)

### 💡 Interpretación del análisis

Lo que acabamos de hacer es **el caso de uso #1 de Delta Sharing en proyectos SAP**:

- **Ventas reales** vienen de SAP S/4HANA → Bronze → Silver → Gold (en este workspace)
- **Presupuesto** viene del equipo FP&A vía Delta Sharing (sin que ellos repliquen las ventas)
- **Tasa de cambio** vendría del Banco de la República vía Delta Sharing (otro share)
- El **análisis** se hace en este workspace usando NUESTRO cómputo

En BDC sería:
- Ventas reales → publicadas por SAP como Data Product
- Presupuesto → tabla propia en este workspace (resultado del análisis)
- El cruce vive donde sea más eficiente — ambos lados ven datos frescos

> *"La bidireccionalidad de BDC permite que el resultado de este análisis (varianza por org/mes) regrese a SAP como nuevo data product."*


---
## SECCIÓN 4 — Caso 2: Cruce con datos externos compartidos

Ahora cruzamos clientes SAP con datos demográficos del share `samples.bakehouse` — simulando un enriquecimiento desde un partner externo via Delta Sharing real.


In [0]:
# ── Cruzar gold_customer_360 (SAP) con datos externos vía Delta Share ──
print("=== Clientes SAP enriquecidos con datos externos (Delta Sharing real) ===\n")

spark.sql("""
    SELECT
        g.customer_id                              AS cliente_sap,
        g.customer_name                            AS nombre_sap,
        g.customer_tier                            AS tier_sap,
        ROUND(g.total_revenue, 2)                  AS revenue_sap,
        -- Datos del share externo (samples.bakehouse)
        s.city                                     AS ciudad_externa,
        s.state                                    AS estado_externo,
        s.continent                                AS continente
    FROM laboratory_sap_dev.sap_course.gold_customer_360  g
    LEFT JOIN samples.bakehouse.sales_customers           s
      ON g.customer_id = CAST(s.customerID AS INT)
    WHERE g.customer_tier IN ('PLATINUM','GOLD')
    ORDER BY g.total_revenue DESC
    LIMIT 15
""").show(truncate=False)

### 💡 Lo que demuestra este JOIN

```
Tabla SAP local (gold_customer_360)    +    Tabla via Delta Sharing (samples.bakehouse)
        ↓                                              ↓
    JOIN en SQL estándar — Spark no distingue entre tablas locales y compartidas
        ↓
    Resultado: análisis cruzado sin mover ni replicar datos
```

Este patrón es el corazón de la arquitectura Lakehouse moderna:
- **Datos sensibles internos** (SAP) → quedan en el workspace de Summa
- **Datos externos** (partners, BDC, datos públicos) → se consumen via Delta Sharing
- **El análisis** vive donde está el cómputo del consumidor

> *"En el módulo siguiente verán cómo SAP BDC convierte sus tablas BKPF, VBAK, MARA en data products listos para este tipo de cruce — sin necesidad de extractor ni replicación."*


---
## SECCIÓN 5 — Conexión con SAP BDC

Lo que acabamos de hacer es **arquitectónicamente idéntico** a la integración SAP BDC + Databricks:


In [0]:
print("""
COMPARACIÓN: Delta Sharing hoy vs SAP BDC en producción
═══════════════════════════════════════════════════════════════════════

┌──────────────────────────────┬─────────────────────────────────────┐
│ DELTA SHARING HOY            │ SAP BDC EN PRODUCCIÓN               │
├──────────────────────────────┼─────────────────────────────────────┤
│ samples.bakehouse            │ sap_bdc.sales.I_SalesOrder          │
│ samples.tpch                 │ sap_bdc.finance.I_JournalEntry      │
│ (shared by Databricks)       │ (shared by SAP BDC)                 │
│                              │                                     │
│ Lo configura: Databricks     │ Lo configura: BDC Connect (1 vez)   │
│ Catálogo: samples            │ Catálogo: sap_bdc                   │
│                              │                                     │
│ Auth: automática             │ Auth: mTLS + OIDC                   │
│ Token: gestionado por DB     │ Token: gestionado por BDC           │
│                              │                                     │
│ Para conectar tu propio      │ Para conectar tu workspace:         │
│ share externo necesitas:     │ - SAP configura BDC Connect         │
│ - Metastore admin            │ - One-time setup                    │
│ - External OpenSharing ON    │ - Aparece como catálogo en UC       │
│                              │                                     │
│ El problema que vimos:       │ La solución que ofrece BDC:         │
│ Free Edition con cuenta      │ Foundation Services gestiona TODA   │
│ personal no permite cambiar  │ la configuración del metastore por  │
│ el metastore (System user)   │ ti — solo consumes data products    │
└──────────────────────────────┴─────────────────────────────────────┘

→ POR ESO BDC ES TAN VALIOSO PARA PROYECTOS SAP:
   Resuelve el setup operativo (metastore, recipients, auth)
   y entrega data products curados con semántica de negocio.
""")

---
## Resumen del Módulo 6

✅ **Concepto** — Delta Sharing: protocolo abierto, datos sin moverlos, zero-copy

✅ **Lado proveedor** — Vimos `sap_analytics_share` con tres dominios (finance, sales, customers) y los comandos SQL completos

✅ **Lado receptor** — Consumimos `samples` (un Delta Share real de Databricks) y vimos que se comporta como cualquier tabla local

✅ **Caso de negocio real**:
   - Análisis Real vs Budget cruzando gold_sales_kpis × budget de FP&A
   - Enriquecimiento de gold_customer_360 con datos externos vía Delta Sharing real

✅ **Conexión con BDC** — Lo que hicimos es exactamente la experiencia BDC + Databricks: SAP publica data products, nosotros consumimos sin ETL

### Para la próxima clase
- **Módulo 7**: BDC Connect en profundidad — cómo se configura, cómo se monitorea
- **Lab final**: simular el ciclo bidireccional — datos SAP entran, resultado ML regresa

> *"Hoy fueron consumidores de un share real. La próxima clase entendemos qué hace BDC del lado SAP para que el share llegue sin que ustedes muevan un dedo en el extractor."*
